# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Omahcy/FlyRank_Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [11]:
import os, getpass, duckdb
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Token: env var, then Colab Secret HF_TOKEN, then a prompt as a last resort.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Full daily table (the sample was too thin: no page reached 100 impressions in a window).
DATA_PATH = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'

# Two 30-day windows per page, anchored on the latest date in the table.
df = con.sql(f"""
WITH b AS (SELECT MAX(report_date) AS end_d FROM read_parquet('{DATA_PATH}'))
SELECT f.client_hash_id, f.content_hash_id,
  SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
  SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30
FROM read_parquet('{DATA_PATH}') f, b
WHERE f.report_date > b.end_d - INTERVAL 60 DAY
GROUP BY 1, 2
HAVING imp_prev30 >= 100
""").df()

if len(df) == 0:
    raise ValueError('No pages met the 100-impression minimum. Check DATA_PATH and the date range.')

df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)
print('pages:', len(df), '  decline rate:', round(df['is_declining'].mean(), 3))

# Leakage test: can the label's own input (imp_prev30) predict the label, for unseen clients?
X, y, g = df[['imp_prev30']], df['is_declining'], df['client_hash_id']
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0).split(X, y, g))
m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X.iloc[tr], y.iloc[tr])
auc = roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1])
print('ROC-AUC using imp_prev30 alone, unseen clients:', round(auc, 3), '(0.5 = chance)')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 111247   decline rate: 0.643
ROC-AUC using imp_prev30 alone, unseen clients: 0.502 (0.5 = chance)


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing values | Available before the prediction moment? |
|---|---|---|---|
| imp_prev30 | Impressions in the 30 days before the last 30 | None after the 100-impression minimum | Yes, but the label is computed from it |
| visible_queries | Distinct queries that showed the page (90-day table) | Pages without query rows were dropped | No: the 90-day window overlaps the label window |
| rare_share | Share of impressions from rare queries | Same | No, same overlap |
| anon_share | Share of impressions from hidden (anonymized) queries | Same | No, same overlap |
| top_query_share | Share of impressions held by the top query | Same | No, same overlap |

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Test run in this notebook.** The label is `is_declining`: last-30-day impressions below 80%
of the previous 30 days. Its own input, `imp_prev30`, could simply recover it, so I trained a
small decision tree on `imp_prev30` alone and scored it on clients held out of training. On
111,247 pages (decline rate 0.643) the ROC-AUC was 0.502, which is chance. The label's
arithmetic is therefore not being recovered through that feature.

**Other checks from my capstone.**
- A random split lets one client appear in train and test. A per-client split lowered
  accuracy from 0.650 to 0.614, so part of the first result was client-specific.
- The query features come from a 90-day table that overlaps the label window. This cannot
  be removed, so results are described as contemporaneous associations.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- **imp_last30:** it defines the label.
- **clk_last30, pos_last30:** measured in the same window as the outcome.
- **client_hash_id, content_hash_id:** pseudonymous IDs, used only for grouping and splits,
  never as features (per DATA_USE.md).
- **Pages with under 100 impressions in the earlier window:** too little history for a
  percentage change.
- **Pages with missing query features:** dropped before modelling.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.